In [ ]:
import pandas as pd
import requests
import logging
from sklearn.metrics import classification_report, accuracy_score
from med_llm_utils.medcat_llm_validator.validator import validate_annotations
from dotenv import load_dotenv
import os

# 1. Setup Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

config = type('Config', (), {'db_engine': None})()

# 2. Load the Test Dataset
csv_path = "../../../med_llm_utils/medcat_llm_validator/llm_test_dataset.csv"
try:
    df_test = pd.read_csv(csv_path, sep=',', quotechar='"', on_bad_lines='warn') #, nrows=5)
    print(f"✅ Successfully loaded {len(df_test)} records.")
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    df_test = pd.DataFrame()

df_test = df_test.sample(5)

# 3. Enhanced Ollama Caller
def create_ollama_caller(model_name: str, host: str, timeout: int, is_reasoning: bool):
    def ollama_caller(prompt: str) -> str:
        url = f"http://{host}:11434/api/generate"
        options = {"temperature": 0.0, "stop": ["---END---"]}
        if is_reasoning:
            options.update({"num_ctx": 8192, "num_predict": 4096})

        payload = {"model": model_name, "prompt": prompt, "stream": False, "options": options}
        
        try:
            response = requests.post(url, json=payload, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            
            # --- FIX: CAPTURE THE MONOLOGUE ---
            # Ollama reasoning models often put thinking in a separate 'reasoning' field
            resp_text = data.get("response", "")
            reasoning = data.get("reasoning", "")
            
            if reasoning:
                # Merge them so the validator sees the 'unstripped' raw output
                return f"<think>\n{reasoning}\n</think>\n{resp_text}"
            return resp_text
        except Exception as e:
            logger.error(f"Ollama call failed: {e}")
            return f"ERROR: {str(e)}"
            
    return ollama_caller

# --- CONFIGURATION ---
# Load variables from .env
load_dotenv()

# Access the variable
remote_server_ip = os.getenv("OLLAMA_REMOTE_IP", "127.0.0.1") # Fallback to localhost if not found
local_server_ip = os.getenv("OLLAMA_LOCAL_IP", "127.0.0.1")

# Access the variable
use_reasoning = True 
model = "qwen3:30b-thinking" if use_reasoning else "gemma2:9b"
host = remote_server_ip if use_reasoning else local_server_ip
print(f"Connecting to: {host}")
timeout = 300
my_caller = create_ollama_caller(model, host, timeout, use_reasoning)

# 4. Run Validation
if not df_test.empty:
    print("\n🚀 Running Validation...")

    validated_df = validate_annotations(
        df=df_test,
        db_engine=config.db_engine,
        llm_caller=my_caller,
        text_column="text_content",
        is_reasoning_model=use_reasoning,
        use_concise_prompts=True,         # Fast but structured
        dump_full_reasoning_response=True, # Moves unstripped monologue to 'llm_reason'
        n_workers=1,
        skip_availability_check=True,
        debug_mode=True 
    )

    # 5. Performance Metrics (with mapping fix)
    eval_df = validated_df[validated_df["llm_status"].isin(["Confirmed", "Rejected", "Uncertain"])].copy()

    if not eval_df.empty:
        print("\n" + "="*60 + "\nPERFORMANCE METRICS\n" + "="*60)
        
        # Mapping Ground Truth abbreviations to LLM full-word outputs
        gt_mapping = {
            'y': 'true', 'n': 'false', 
            'p': 'present', 'pa': 'past', 'f': 'future',
            'o': 'other', 'pat': 'patient', 'rel': 'relative'
        }

        # Overall Adjudication
        y_true = eval_df['ground_truth_status'].astype(str).str.title()
        y_pred = eval_df['llm_status'].astype(str).str.title()
        print(f"Overall Adjudication Accuracy: {accuracy_score(y_true, y_pred):.2%}")
        print(classification_report(y_true, y_pred, zero_division=0))

        # Task-specific metrics
        llm_cols = [c for c in validated_df.columns if c.startswith("llm_") and not c.startswith("llm_status_") and c not in ["llm_status", "llm_reason", "llm_raw_response"]]
        for col in llm_cols:
            task_id = col.replace("llm_", "")
            gt_col = f"gt_{task_id}"
            if gt_col in eval_df.columns:
                print(f"\n[TASK: {task_id.upper()}]")
                y_t = eval_df[gt_col].astype(str).str.lower().map(lambda x: gt_mapping.get(x, x))
                y_p = eval_df[col].astype(str).str.lower()
                valid = (y_t != 'unknown') & (y_t != 'n/a') & (y_p != 'unknown')
                if valid.any():
                    print(f"Accuracy: {accuracy_score(y_t[valid], y_p[valid]):.2%}")
                    print(classification_report(y_t[valid], y_p[valid], zero_division=0))
    else:
        print("\n⚠️ No evaluation data.")


In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
validated_df

In [ ]:
from med_llm_utils.medcat_llm_validator.plotting import (
    plot_llm_validation_audit, 
    plot_per_task_details,  
    plot_accuracy_by_concept, 
    plot_reasoning_verbosity
)


In [ ]:
# 1. Main Dashboard (Confusion Matrix and Task Breakdown)
plot_llm_validation_audit(validated_df)

In [ ]:
# 2. Per-Task Detailed Confusion Matrices
plot_per_task_details(validated_df)


In [ ]:
# 4. Top 15 Hardest/Easiest Concepts
plot_accuracy_by_concept(validated_df, top_n=15)


In [ ]:
# 4. Top 15 Hardest/Easiest Concepts
plot_accuracy_by_concept(validated_df, top_n=15)


In [ ]:
# 5. Reasoning analysis (See how much it 'thinks' for Rejections vs Confirmations)
plot_reasoning_verbosity(validated_df)
